# EDA and Model Benchmark

This notebook demonstrates: feature extraction, visualization, baseline models, hyperparameter tuning, and clustering.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from src.features import extract_features
from src.tuning import grid_search_classic
sns.set()
%matplotlib inline


In [ ]:
# Adjust data_dir to your local path
DATA_DIR = "../data"
rows = []
for cls in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(p):
        continue
    for f in sorted(os.listdir(p)):
        if not f.lower().endswith((".mp3", ".wav")):
            continue
        fp = os.path.join(p, f)
        feats, names = extract_features(fp)
        row = dict(zip(names, feats.tolist()))
        row["file_path"] = fp
        row["label"] = cls
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("features.csv", index=False)
print("Saved features.csv with", len(df), "rows")

In [ ]:
# Basic class distribution
print(df["label"].value_counts())
plt.figure(figsize=(8, 4))
sns.countplot(y="label", data=df)
plt.title("Class distribution")
plt.show()

In [ ]:
# PCA visualization
X = df.drop(columns=["file_path", "label"]).values
le = LabelEncoder()
y = le.fit_transform(df["label"].values)
X_s = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_p = pca.fit_transform(X_s)
plt.figure(figsize=(8, 6))
plt.scatter(X_p[:, 0], X_p[:, 1], c=y, cmap="tab10", alpha=0.7)
plt.title("PCA of features")
plt.show()

In [ ]:
# Quick GridSearch (may take time). Results saved under `experiments/` and `models/`.
res = grid_search_classic(
    X, y, cv=3, experiments_dir="experiments", models_dir="models"
)
for k, v in res.items():
    print(k, v["best_score"], v["best_params"])